# ????????? ML/RL KuCoin Demo (NEAR Spot + Perp)

???????? ?????? ???:
- ???????? ?????? KuCoin (OHLCV),
- ML-????????? (`ARIMA` + `GARCH`),
- ??????? ??????? ??????? ? ?????,
- toy RL-??????????? ??????????????,
- ???????? ?????????? ????????? ? ???????????? ??????.


## ????????????
- ??????? ?? ??????? API-????? ? ??????? ????????.
- ??????????? ?????? ?????????? ?????????.
- ???? ???? ??? ???????????, ????? ???????? ??? ? KuCoin UI.


In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
sys.path.insert(0, str(ROOT / 'src'))

from delta_bot.config import load_config
from delta_bot.kucoin_client import KuCoinRestClient
from delta_bot.signal import spot_candle_type_from_minutes
from delta_bot.policy import compute_target_positions
from delta_bot.reward import RewardInputs, compute_reward

cfg = load_config(ROOT / 'config' / 'micro_near_v1.json')
client = KuCoinRestClient.from_env()
cfg

In [ ]:
# ??? 1. ????????? spot-????? (????????? endpoint)
candle_type = spot_candle_type_from_minutes(cfg.timing.data_tf_minutes)
spot_rows = client.get_spot_candles(cfg.instruments.spot_symbol, candle_type)
spot_rows = sorted(spot_rows, key=lambda x: int(x[0]))

df = pd.DataFrame(spot_rows, columns=['ts', 'open', 'close', 'high', 'low', 'volume', 'turnover'])
for c in ['open', 'close', 'high', 'low', 'volume', 'turnover']:
    df[c] = pd.to_numeric(df[c], errors='coerce')
df['ts'] = pd.to_datetime(df['ts'].astype(int), unit='s', utc=True)
df['ret'] = np.log(df['close'] / df['close'].shift(1))
df.tail(5)

In [ ]:
# ??? 2. ML-????????: ARIMA ??? ??????????, GARCH ??? ?????????????
from statsmodels.tsa.arima.model import ARIMA

ret = df['ret'].dropna()
arima = ARIMA(ret, order=(2, 0, 2)).fit()
ret_hat = float(arima.forecast(1).iloc[0])

try:
    from arch import arch_model
    garch = arch_model(ret * 100, vol='Garch', p=1, q=1, rescale=False).fit(disp='off')
    sigma_hat = float(np.sqrt(garch.forecast(horizon=1).variance.iloc[-1, 0]) / 100)
except Exception:
    sigma_hat = float(ret.rolling(20).std().iloc[-1])

ret_hat, sigma_hat

In [ ]:
# ??? 3. ??????? ??????? ?? ML-?????? (? ????????????? ??? ???????? futures)
spot_price = float(df['close'].iloc[-1])
target = compute_target_positions(
    ret_hat=ret_hat,
    sigma_hat=sigma_hat,
    spot_price=spot_price,
    policy_cfg=cfg.policy,
    instr_cfg=cfg.instruments,
)
target

In [ ]:
# ??? 4. Toy RL-????: Q-learning ?? ?????????? hedge-?????????
actions = np.array([-2, -1, 0, 1, 2])  # contracts delta
Q = np.zeros((21, len(actions)))  # simple discrete net-delta state bins
alpha, gamma, eps = 0.1, 0.95, 0.2

def to_state(net_delta_notional):
    v = int(np.clip(round(net_delta_notional * 10), -10, 10))
    return v + 10

net_delta_notional = 0.0
c_prev = 0
for _ in range(500):
    s = to_state(net_delta_notional)
    if np.random.rand() < eps:
        ai = np.random.randint(len(actions))
    else:
        ai = int(np.argmax(Q[s]))
    dc = int(actions[ai])
    c_now = np.clip(c_prev + dc, -12, 12)

    # ????????????? reward-?????? ??? ????????????
    r = -0.02 * abs(c_now - c_prev) - 0.15 * abs(net_delta_notional)

    # ????????? ??????? ?????????
    net_delta_notional = float(np.clip(net_delta_notional + np.random.randn() * 0.05, -1.0, 1.0))
    s2 = to_state(net_delta_notional)
    Q[s, ai] = Q[s, ai] + alpha * (r + gamma * np.max(Q[s2]) - Q[s, ai])
    c_prev = int(c_now)

Q[:3, :], Q.shape

In [ ]:
# ??? 5. ?????? ??????? reward ?? ????? ????
ri = RewardInputs(
    pnl_usdt=0.0036,
    fee_usdt=0.0005,
    funding_usdt=0.0001,
    net_delta_notional_usdt=0.0,
    delta_contracts=0,
    drawdown_fraction=0.012,
)
compute_reward(ri, cfg.reward)

In [ ]:
# ??? 6. ????????????? ??????????: shadow ? live
# Shadow (??? ???????? ???????):
#   set PYTHONPATH=src
#   python -m delta_bot.live --mode shadow --config config/micro_near_v1.json --state-file .runtime/bot_state.json
#
# Live (???????? ??????):
#   set KUCOIN_API_KEY=...
#   set KUCOIN_API_SECRET=...
#   set KUCOIN_API_PASSPHRASE=...
#   set KUCOIN_KEY_VERSION=2
#   python -m delta_bot.live --mode live --config config/micro_near_v1.json --state-file .runtime/bot_state.json
print('??????? ?????: ????? ?????????????? state JSON ? ????????? ???????????? ??????')

## 8. ??????? ?????????? ????????? ??? ????????????? RL-????

??? ? `lecture15`: ? ????? ???????? ????????? ????????? JSON ? ????????? ?????????? ????? ? ?????????? ????????.
?????? ???? ???? ???????? ? `run_trade_signal.py` / `trade_signal_executor_kucoin.py`.


In [ ]:
# ??????? ?????????? ????????? ????? ??? ????????????? ???????
import json
import shutil
from datetime import datetime, timezone

spot_t = client.get_spot_ticker(cfg.instruments.spot_symbol)
fut_t = client.get_futures_ticker(cfg.instruments.futures_symbol)

state_payload = {
    "ts_utc": datetime.now(timezone.utc).isoformat(),
    "exchange": "kucoin",
    "spot_symbol": cfg.instruments.spot_symbol,
    "futures_symbol": cfg.instruments.futures_symbol,
    "data_tf_minutes": cfg.timing.data_tf_minutes,
    "ret_hat": float(ret_hat),
    "sigma_hat": float(sigma_hat),
    "spot_price": float(spot_t["price"]),
    "futures_price": float(fut_t["price"]),
    "target_spot_qty": float(target.target_spot_qty),
    "target_futures_contracts": int(target.target_futures_contracts),
    "target_net_delta_base": float(target.target_net_delta_base),
}

out_dir = ROOT / "reports" / "kucoin_rl"
out_dir.mkdir(parents=True, exist_ok=True)
state_path = out_dir / "latest_forecast_signal_kucoin_rl.json"
state_path.write_text(json.dumps(state_payload, ensure_ascii=False, indent=2), encoding="utf-8")

# ??????????? ???????? ? Downloads, ????? ?????? ??? auto-discovery ??? ? lecture15
downloads_candidates = [Path.home() / "Downloads", Path.home() / "????????"]
for d in downloads_candidates:
    if d.exists():
        try:
            shutil.copy2(state_path, d / state_path.name)
        except Exception:
            pass

print("State JSON saved:", state_path)
print(json.dumps(state_payload, ensure_ascii=False, indent=2))
